# Reversal study - Colab pilot

A scaled-down run of `scripts/run_reversal_study.py`: 3 agents (baseline / emotional / yoked)
x 5 seeds on `shield_trap_easy`, sized to finish inside one Colab session.

**Why `--reversals 4` and not 2.** The pre-registered primary metric is
*mean episodes-to-recovery over reversals 2..R* - reversal 1 is excluded by design
(`utils/reversal_analysis.py`), because it is confounded with the end of acquisition.
Phase A acquires under `protective`; each reversal block *starts* with a flip, so the
blocks run `block1 = non_protective, block2 = protective, block3 = non_protective, ...`
Dropping block 1 therefore drops the only non_protective block until R reaches 3:

| R | scored blocks | contingencies actually measured |
|---|---------------|---------------------------------|
| 2 | block 2 only  | protective only (the one it already learned) - **degenerate** |
| 3 | blocks 2,3    | protective + non_protective - bare minimum |
| 4 | blocks 2,3,4  | 2x protective + 1x non_protective - recommended |

Each extra reversal costs only `reversal_period` episodes (~1 min/cell), so R=4 is
cheap insurance. Set `REVERSALS = 2` below if you want the degenerate version anyway.

**Resumable.** The study writes into Google Drive and skips any cell whose
`reversal_manifest.json` exists. If Colab disconnects, re-run the setup cells and the
run cell with the *same* `STUDY_NAME` - it picks up where it stopped.


## 1. Environment

In [ ]:
!nvidia-smi -L || echo "NO GPU - set Runtime > Change runtime type > T4 GPU"


In [ ]:
REPO_URL = "https://github.com/zhihankshi/emotion_dqn.git"
BRANCH   = "main"

import os, subprocess, sys
if not os.path.isdir("/content/emotion_dqn"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/emotion_dqn"], check=True)
else:
    subprocess.run(["git", "-C", "/content/emotion_dqn", "pull"], check=True)

os.chdir("/content/emotion_dqn")
sys.path.insert(0, "/content/emotion_dqn")
!pip -q install -r requirements.txt
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/emotion_dqn_runs"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("results ->", DRIVE_ROOT)


## 2. Sanity checks

Env mechanics + per-maze reward accounting. Run before training.

In [ ]:
!python tests/sanity_checks.py


## 3. Configuration

`STUDY_NAME` is the resume key - keep it fixed across reconnects, change it to start fresh.

In [ ]:
MAZE        = "shield_trap_easy"   # small + max_steps 40; fastest maze with a shield/trap contingency
AGENTS      = "baseline,emotional,yoked"
SEEDS       = [1, 2, 3, 4, 5]
REVERSALS   = 4                    # see note at top; 2 makes the primary metric degenerate
PERIOD      = 150                  # episodes per reversal block (K)

# Acquisition (Phase A)
CRITERION_RATE   = 0.80
CRITERION_WINDOW = 50
MAX_ACQ          = 600             # cap; cells that never hit criterion are dropped from the
                                   # primary metric, so don't set this too tight

# Held identical across all three arms - this is the matched-comparison invariant
EPS_FLOOR   = 0.05
EPS_DECAY   = 120
BUFFER      = 12000
NON_PROT    = -60.0                # trap_with_shield under the non_protective contingency
IMAGE_SIZE  = 64
NETWORK     = "standard"

STUDY_NAME  = f"colab_reversal_{MAZE}_R{REVERSALS}K{PERIOD}_s{len(SEEDS)}"
STUDY_DIR   = f"{DRIVE_ROOT}/{STUDY_NAME}"

N_CELLS = len(AGENTS.split(",")) * len(SEEDS)
print(f"{N_CELLS} cells -> {STUDY_DIR}")
print(f"~{MAX_ACQ} acquisition + {REVERSALS * PERIOD} reversal episodes per cell (worst case)")


## 4. Timing probe

One throwaway baseline cell at tiny settings, to extrapolate the wall clock before
committing. Writes to `/content` scratch, not Drive.

In [ ]:
import time, shutil
shutil.rmtree("/content/_probe", ignore_errors=True)

t0 = time.time()
!python scripts/run_reversal_study.py \
    --maze {MAZE} --agents baseline --seeds 1 \
    --reversals 1 --reversal_period 40 \
    --criterion_rate {CRITERION_RATE} --criterion_window 20 \
    --max_acquisition_episodes 80 \
    --epsilon_floor {EPS_FLOOR} --epsilon_decay_episodes 40 \
    --buffer_size {BUFFER} --non_protective_trap {NON_PROT} \
    --image_size {IMAGE_SIZE} --network_size {NETWORK} \
    --log_dir /content/_probe --study_name probe
probe_s = time.time() - t0

import glob
sched = glob.glob("/content/_probe/probe/*/*/*/reversal_schedule.csv")
n_eps = sum(1 for _ in open(sched[0])) - 1 if sched else 120
rate  = n_eps / probe_s
print(f"\nprobe: {n_eps} episodes in {probe_s:.0f}s = {rate:.1f} eps/s (baseline arm)")

# emotional/yoked carry the mood update; assume ~1.4x slower as a rough guard
worst_eps = (MAX_ACQ + REVERSALS * PERIOD) * N_CELLS
print(f"worst-case {worst_eps} episodes -> {worst_eps / rate / 3600:.1f}h (baseline rate)"
      f" / {worst_eps * 1.4 / rate / 3600:.1f}h (mood-arm rate)")
print("Acquisition usually stops well before the cap, so expect materially less.")


## 5. Run the study

Order matters: emotional runs finish before yoked ones, because each yoked cell
consumes the mood trace of the emotional run at a *different* seed. Re-running this
cell after a disconnect skips completed cells.

In [ ]:
SEEDS_ARG = " ".join(str(s) for s in SEEDS)

!python scripts/run_reversal_study.py \
    --maze {MAZE} --agents {AGENTS} --seeds {SEEDS_ARG} \
    --reversals {REVERSALS} --reversal_period {PERIOD} \
    --criterion_rate {CRITERION_RATE} --criterion_window {CRITERION_WINDOW} \
    --max_acquisition_episodes {MAX_ACQ} \
    --epsilon_floor {EPS_FLOOR} --epsilon_decay_episodes {EPS_DECAY} \
    --buffer_size {BUFFER} --non_protective_trap {NON_PROT} \
    --yoked_mode replay_trace \
    --image_size {IMAGE_SIZE} --network_size {NETWORK} \
    --log_dir {DRIVE_ROOT} --study_name {STUDY_NAME}


## 6. Analysis

`mood_dip_pass_rate` is the gate: if M does not measurably drop after a reversal,
the mood mechanism is not engaging and no adaptation-speed difference can be
attributed to it - read that line before the contrasts.

In [ ]:
from utils.reversal_analysis import analyze_study, print_report
import json

analysis = analyze_study(STUDY_DIR, window=20, recovery_fraction=0.80)
print_report(analysis)

with open(f"{STUDY_DIR}/analysis.json", "w") as f:
    json.dump(analysis, f, indent=2, default=str)


In [ ]:
# Per-seed primary metric, so you can see whether a group mean is one seed's doing.
for agent, g in analysis["groups"].items():
    print(f"{agent:<11} n={g['n_runs']}  excluded(non-acquirer)={g['n_excluded_non_acquirers']}")
    print(f"            per-seed: {g['primary_per_seed']}")


## 7. Figures

In [ ]:
!python visualize_reversal.py --study_dir {STUDY_DIR} --output_dir {STUDY_DIR}/figures --window 20

from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{STUDY_DIR}/figures/*.png")):
    print(p); display(Image(p))


## Notes

- **n=5 per arm is a pilot, not a test.** With 3 scored blocks per run the primary
  metric is a per-run mean over 3 numbers, and the group contrast has 5 runs a side -
  enough to see whether the pipeline produces a sane, non-degenerate signal and to get
  a variance estimate for powering the cluster run. Read `cohens_d` as an effect-size
  sketch, not evidence.
- **Non-acquirers are dropped** from the primary metric by default. If
  `n_excluded_non_acquirers` is more than ~1 per arm, raise `MAX_ACQ` rather than
  reporting the surviving subset.
- **Do not "fix"** these at a reversal: epsilon is never bumped, the buffer is never
  flushed, the target net is never reset, mood is never reset. They are the design.
- To scale up on the cluster, the same command with `--seeds $(seq 1 20)`,
  `--reversals 8 --reversal_period 400` is the full study.
